# Исследование перекоса данных

Исходный INNER JOIN просмотров с каталогом, диагностика распределения `element_uid`, Broadcast JOIN и Salt Join

In [1]:
import math
import time
from pathlib import Path

import pandas as pd
from pyspark.sql import SparkSession, functions as F, types as T

spark = (SparkSession.builder
         .appName("Spark_Skew_Join")
         .master("spark://spark-master:7077")
         .getOrCreate())

spark.conf.set("spark.sql.session.timeZone", "UTC")

### Загрузка данных

In [2]:
transactions_path = "/home/jovyan/work/data/task2/skew_transactions.csv"
catalogue_path = "/home/jovyan/work/data/task2/catalogue.json"

transactions_schema = T.StructType([
    T.StructField("user_uid", T.LongType()),
    T.StructField("element_uid", T.LongType()),
    T.StructField("watched_time", T.LongType())
])

df_transactions = (
    spark.read
    .option("header", "true")
    .schema(transactions_schema)
    .csv(transactions_path)
)

catalogue_item_schema = T.StructType([
    T.StructField("type", T.StringType()),
    T.StructField("availability", T.ArrayType(T.StringType())),
    T.StructField("duration", T.IntegerType()),
    T.StructField("feature_1", T.DoubleType()),
    T.StructField("feature_2", T.DoubleType()),
    T.StructField("feature_3", T.DoubleType()),
    T.StructField("feature_4", T.DoubleType()),
    T.StructField("feature_5", T.DoubleType()),
    T.StructField("attributes", T.ArrayType(T.IntegerType()))
])

# Каталог — один JSON-объект с element_uid в качестве ключей
catalogue_json = Path(catalogue_path).read_text(encoding="utf-8")
df_catalogue_json = spark.createDataFrame([(catalogue_json,)], ["json"])
catalogue_map_schema = T.MapType(T.StringType(), catalogue_item_schema)

df_catalogue = (
    df_catalogue_json
    .select(F.from_json("json", catalogue_map_schema).alias("items"))
    .select(F.explode("items").alias("element_uid", "item"))
    .select(
        F.col("element_uid").cast("long").alias("element_uid"),
        "item.*"
    )
)

df_transactions.printSchema()
df_catalogue.printSchema()

root
 |-- user_uid: long (nullable = true)
 |-- element_uid: long (nullable = true)
 |-- watched_time: long (nullable = true)

root
 |-- element_uid: long (nullable = true)
 |-- type: string (nullable = true)
 |-- availability: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- duration: integer (nullable = true)
 |-- feature_1: double (nullable = true)
 |-- feature_2: double (nullable = true)
 |-- feature_3: double (nullable = true)
 |-- feature_4: double (nullable = true)
 |-- feature_5: double (nullable = true)
 |-- attributes: array (nullable = true)
 |    |-- element: integer (containsNull = true)



### Диагностика распределения ключей
Для поиска перекошенных ключей используются два условия: частота не менее чем в 10 раз выше медианной и доля не менее 1% всех просмотров.

In [3]:
transaction_count = df_transactions.count()
null_transaction_keys = df_transactions.filter(F.col("element_uid").isNull()).count()

catalogue_count = df_catalogue.count()
null_catalogue_keys = df_catalogue.filter(F.col("element_uid").isNull()).count()
duplicate_catalogue_keys = (
    df_catalogue.groupBy("element_uid")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

df_key_frequency = (
    df_transactions
    .filter(F.col("element_uid").isNotNull())
    .groupBy("element_uid")
    .count()
)

non_null_transaction_count = (
    df_key_frequency.agg(F.sum("count").alias("total")).first()["total"] or 0
)
median_frequency = (
    df_key_frequency
    .agg(F.expr("percentile_approx(count, 0.5)").alias("median"))
    .first()["median"] or 0
)

df_key_frequency = df_key_frequency.withColumn(
    "share_pct",
    F.round(F.col("count") * 100 / F.lit(non_null_transaction_count), 3)
)

print("Строк просмотров:", transaction_count)
print("Просмотры с NULL element_uid:", null_transaction_keys)
print("Строк каталога:", catalogue_count)
print("NULL element_uid в каталоге:", null_catalogue_keys)
print("Дубли element_uid в каталоге:", duplicate_catalogue_keys)
print("Медианная частота ключа:", median_frequency)

df_key_frequency.orderBy(F.desc("count")).show(20, truncate=False)

Строк просмотров: 30070692
Просмотры с NULL element_uid: 0
Строк каталога: 10200
NULL element_uid в каталоге: 0
Дубли element_uid в каталоге: 0
Медианная частота ключа: 179
+-----------+--------+---------+
|element_uid|count   |share_pct|
+-----------+--------+---------+
|2714       |20496000|68.159   |
|747        |61272   |0.204    |
|2639       |61119   |0.203    |
|3783       |55228   |0.184    |
|2245       |52942   |0.176    |
|6127       |52168   |0.173    |
|10061      |48993   |0.163    |
|3916       |48243   |0.16     |
|8771       |47480   |0.158    |
|3336       |45646   |0.152    |
|7079       |43872   |0.146    |
|3567       |42278   |0.141    |
|6195       |42192   |0.14     |
|4141       |41704   |0.139    |
|1016       |41539   |0.138    |
|7185       |39595   |0.132    |
|1004       |38929   |0.129    |
|9467       |37207   |0.124    |
|72         |36080   |0.12     |
|1521       |35168   |0.117    |
+-----------+--------+---------+
only showing top 20 rows



### Исходный INNER JOIN

In [4]:
df_join_original = df_transactions.join(
    df_catalogue,
    on="element_uid",
    how="inner"
)

df_join_original.explain("formatted")

== Physical Plan ==
AdaptiveSparkPlan (15)
+- Project (14)
   +- SortMergeJoin Inner (13)
      :- Sort (4)
      :  +- Exchange (3)
      :     +- Filter (2)
      :        +- Scan csv  (1)
      +- Sort (12)
         +- Exchange (11)
            +- Project (10)
               +- Filter (9)
                  +- Generate (8)
                     +- Project (7)
                        +- Filter (6)
                           +- Scan ExistingRDD (5)


(1) Scan csv 
Output [3]: [user_uid#0L, element_uid#1L, watched_time#2L]
Batched: false
Location: InMemoryFileIndex [file:/home/jovyan/work/data/task2/skew_transactions.csv]
PushedFilters: [IsNotNull(element_uid)]
ReadSchema: struct<user_uid:bigint,element_uid:bigint,watched_time:bigint>

(2) Filter
Input [3]: [user_uid#0L, element_uid#1L, watched_time#2L]
Condition : isnotnull(element_uid#1L)

(3) Exchange
Input [3]: [user_uid#0L, element_uid#1L, watched_time#2L]
Arguments: hashpartitioning(element_uid#1L, 200), ENSURE_REQUIREMENTS, [plan_

### Broadcast JOIN
Каталог используется как справочник. Broadcast применим, если его размер позволяет разместить копию на executor.

In [5]:
catalogue_size_mb = Path(catalogue_path).stat().st_size / (1024 ** 2)
print(f"Размер catalogue.json: {catalogue_size_mb:.2f} MB")
print(f"Количество строк каталога: {catalogue_count:,}")
print("Память executor:", spark.sparkContext.getConf().get("spark.executor.memory", "не задано"))

df_join_broadcast = df_transactions.join(
    F.broadcast(df_catalogue),
    on="element_uid",
    how="inner"
)

df_join_broadcast.explain("formatted")

Размер catalogue.json: 2.70 MB
Количество строк каталога: 10,200
Память executor: 6g
== Physical Plan ==
AdaptiveSparkPlan (12)
+- Project (11)
   +- BroadcastHashJoin Inner BuildRight (10)
      :- Filter (2)
      :  +- Scan csv  (1)
      +- BroadcastExchange (9)
         +- Project (8)
            +- Filter (7)
               +- Generate (6)
                  +- Project (5)
                     +- Filter (4)
                        +- Scan ExistingRDD (3)


(1) Scan csv 
Output [3]: [user_uid#0L, element_uid#1L, watched_time#2L]
Batched: false
Location: InMemoryFileIndex [file:/home/jovyan/work/data/task2/skew_transactions.csv]
PushedFilters: [IsNotNull(element_uid)]
ReadSchema: struct<user_uid:bigint,element_uid:bigint,watched_time:bigint>

(2) Filter
Input [3]: [user_uid#0L, element_uid#1L, watched_time#2L]
Condition : isnotnull(element_uid#1L)

(3) Scan ExistingRDD
Output [1]: [json#6]
Arguments: [json#6], MapPartitionsRDD[4] at applySchemaToPythonRDD at NativeMethodAccessorImpl

### Salt Join
Для горячих ключей строки просмотров получают один из 8 salt-бакетов, а соответствующие строки каталога копируются во все 8 бакетов. Восемь бакетов - вдвое больше 4 cores worker. Остальные ключи соединяются обычным JOIN. Так сохраняются все совпадения, но увеличивается объём каталога для горячих ключей.

In [6]:
SKEW_MULTIPLIER = 10
MIN_HOT_KEY_SHARE = 0.01
SALT_BUCKETS = 8

hot_key_threshold = max(
    math.ceil(median_frequency * SKEW_MULTIPLIER),
    math.ceil(non_null_transaction_count * MIN_HOT_KEY_SHARE)
)

df_matching_key_stats = df_key_frequency.join(
    df_catalogue.select("element_uid").distinct(),
    on="element_uid",
    how="inner"
)

df_hot_key_stats = (
    df_matching_key_stats
    .filter(F.col("count") >= hot_key_threshold)
    .orderBy(F.desc("count"))
)

hot_keys = [
    row["element_uid"]
    for row in df_hot_key_stats.select("element_uid").collect()
]

df_hot_keys = spark.createDataFrame(
    [(key,) for key in hot_keys],
    T.StructType([T.StructField("element_uid", T.LongType())])
)

print("Порог частоты перекошенного ключа:", hot_key_threshold)
print("Число перекошенных ключей:", len(hot_keys))
print("Ключи для salting:", hot_keys)
df_hot_key_stats.show(truncate=False)

# Обычные ключи соединяются без соли
df_transactions_regular = df_transactions.join(
    F.broadcast(df_hot_keys), "element_uid", "left_anti"
)
df_catalogue_regular = df_catalogue.join(
    F.broadcast(df_hot_keys), "element_uid", "left_anti"
)
df_join_regular = df_transactions_regular.join(
    df_catalogue_regular, "element_uid", "inner"
)

# Для горячих ключей распределяем строки просмотров по salt-бакетам
df_transactions_hot = (
    df_transactions
    .join(F.broadcast(df_hot_keys), "element_uid", "inner")
    .withColumn(
        "_salt",
        F.pmod(F.monotonically_increasing_id(), F.lit(SALT_BUCKETS)).cast("int")
    )
)

# Распределение горячих ключей по частям
df_transactions_hot \
    .groupBy("_salt") \
    .count() \
    .orderBy("_salt") \
    .show()

# Строки каталога для горячих ключей копируются во все salt-бакеты
df_catalogue_hot = (
    df_catalogue
    .join(F.broadcast(df_hot_keys), "element_uid", "inner")
    .withColumn(
        "_salt",
        F.explode(F.sequence(F.lit(0), F.lit(SALT_BUCKETS - 1)))
    )
)

catalogue_columns = [
    column for column in df_catalogue.columns if column != "element_uid"
]

df_join_hot = (
    df_transactions_hot.alias("t")
    .join(
        df_catalogue_hot.alias("c"),
        (F.col("t.element_uid") == F.col("c.element_uid")) &
        (F.col("t._salt") == F.col("c._salt")),
        "inner"
    )
    .select(
        F.col("t.element_uid").alias("element_uid"),
        F.col("t.user_uid").alias("user_uid"),
        F.col("t.watched_time").alias("watched_time"),
        *[F.col(f"c.`{column}`").alias(column) for column in catalogue_columns]
    )
)

df_join_salted = df_join_regular.unionByName(df_join_hot)

hot_catalogue_rows = df_catalogue.join(
    F.broadcast(df_hot_keys), "element_uid", "inner"
).count()

print("Строк каталога для горячих ключей:", hot_catalogue_rows)
print("Строк каталога после размножения:", hot_catalogue_rows * SALT_BUCKETS)
print("Число salt-бакетов:", SALT_BUCKETS)

df_join_salted.explain("formatted")

Порог частоты перекошенного ключа: 300707
Число перекошенных ключей: 1
Ключи для salting: [2714]
+-----------+--------+---------+
|element_uid|count   |share_pct|
+-----------+--------+---------+
|2714       |20496000|68.159   |
+-----------+--------+---------+

Строк каталога для горячих ключей: 1
Строк каталога после размножения: 8
Число salt-бакетов: 8
== Physical Plan ==
AdaptiveSparkPlan (51)
+- Union (50)
   :- Project (23)
   :  +- SortMergeJoin Inner (22)
   :     :- Sort (9)
   :     :  +- Exchange (8)
   :     :     +- Project (7)
   :     :        +- BroadcastHashJoin LeftAnti BuildRight (6)
   :     :           :- Filter (2)
   :     :           :  +- Scan csv  (1)
   :     :           +- BroadcastExchange (5)
   :     :              +- Filter (4)
   :     :                 +- Scan ExistingRDD (3)
   :     +- Sort (21)
   :        +- Exchange (20)
   :           +- BroadcastHashJoin LeftAnti BuildRight (19)
   :              :- Project (15)
   :              :  +- Filter (1

### Запись результатов и сравнение

In [7]:
OUTPUT_ROOT = "s3a://dwh/task2/join_comparison"
OUTPUT_PATHS = {
    "original": f"{OUTPUT_ROOT}/original",
    "broadcast": f"{OUTPUT_ROOT}/broadcast",
    "salted": f"{OUTPUT_ROOT}/salted"
}

PLAN_DIR = Path("/home/jovyan/work/task2_plans")
PLAN_DIR.mkdir(parents=True, exist_ok=True)

variants = {
    "original": df_join_original,
    "broadcast": df_join_broadcast,
    "salted": df_join_salted
}

def count_parquet_files(path_str):
    hadoop_path = spark._jvm.org.apache.hadoop.fs.Path(path_str)
    filesystem = hadoop_path.getFileSystem(spark._jsc.hadoopConfiguration())
    return sum(
        1 for file_status in filesystem.listStatus(hadoop_path)
        if file_status.isFile() and file_status.getPath().getName().endswith(".parquet")
    )

measurements = []

for variant, df_result in variants.items():
    job_group = f"task2_{variant}"
    spark.sparkContext.setJobGroup(job_group, f"Task 2 JOIN: {variant}")

    start_time = time.perf_counter()
    try:
        df_result.write.mode("overwrite").parquet(OUTPUT_PATHS[variant])
    finally:
        elapsed = time.perf_counter() - start_time
        spark.sparkContext.setLocalProperty("spark.jobGroup.id", None)
        spark.sparkContext.setLocalProperty("spark.job.description", None)

    plan = df_result._jdf.queryExecution().executedPlan().toString()
    (PLAN_DIR / f"{variant}_plan.txt").write_text(plan, encoding="utf-8")

    df_saved = spark.read.parquet(OUTPUT_PATHS[variant])
    row_count = df_saved.count()
    file_count = count_parquet_files(OUTPUT_PATHS[variant])

    measurements.append({
        "variant": variant,
        "time_seconds": round(elapsed, 2),
        "row_count": row_count,
        "parquet_files": file_count,
        "output_path": OUTPUT_PATHS[variant]
    })

    print(f"{variant}: {elapsed:.2f} сек., {row_count:,} строк, {file_count} файлов")

original: 37.30 сек., 30,070,692 строк, 4 файлов
broadcast: 33.04 сек., 30,070,692 строк, 4 файлов
salted: 45.69 сек., 30,070,692 строк, 7 файлов


In [8]:
df_saved_results = {
    variant: spark.read.parquet(path)
    for variant, path in OUTPUT_PATHS.items()
}

df_reference = df_saved_results["original"]

for variant in ["broadcast", "salted"]:
    df_candidate = df_saved_results[variant]

    only_in_reference = df_reference.exceptAll(df_candidate).limit(1).count()
    only_in_candidate = df_candidate.exceptAll(df_reference).limit(1).count()

    print(
        f"{variant}: "
        f"{'результаты совпадают' if only_in_reference == 0 and only_in_candidate == 0 else 'есть различия'}"
    )

    assert only_in_reference == 0 and only_in_candidate == 0

df_comparison = pd.DataFrame(measurements)
display(df_comparison)

broadcast: результаты совпадают
salted: результаты совпадают


,variant,time_seconds,row_count,parquet_files,output_path
0,original,37.30,30070692,4,s3a://dwh/task2/join_comparison/original
1,broadcast,33.04,30070692,4,s3a://dwh/task2/join_comparison/broadcast
2,salted,45.69,30070692,7,s3a://dwh/task2/join_comparison/salted


### Выводы

В исходных данных обнаружен сильный перекос по element_uid. Наиболее частый ключ - 2714 и он встречается 20 496 000 раз, что составляет 68,159% всех просмотров. Его частота примерно в 114 503 раза превышает медианную частоту ключа (179). NULL и дубликаты ключей каталога отсутствуют.

В исходном JOIN используется Sort Merge Join с shuffle по element_uid. Из-за перекоса большая часть строк с ключом 2714 попадает в одну партицию, что может приводить к неравномерному времени выполнения задач.

Для оптимизации выбран Broadcast JOIN. Каталог содержит 10 200 строк, его исходный JSON занимает 2,70 MB, а память executor составляет 6 GB. Физический план подтверждает использование BroadcastHashJoin. Время выполнения составило 33,04 секунды против 37,30 секунды у исходного JOIN.

В Salt Join соль добавляется только для ключа 2714. Используются 8 значений соли. Это позволяет разделить обработку горячего ключа на несколько групп. Строки каталога для этого ключа размножаются с одной до восьми, что добавляет 7 строк. Все исходные совпадения сохраняются.

Несмотря на корректность, Salt Join оказался медленнее: 45,69 секунды. Дополнительные операции разделения данных, размножения каталога, shuffle и объединения результатов создают накладные расходы.

Таким образом, для данного набора данных Broadcast JOIN оказался наиболее подходящим способом оптимизации. Salt Join имеет смысл рассматривать для случаев, когда справочник невозможно эффективно передать через broadcast либо обычное распределение данных продолжает создавать сильный перекос.

Результаты всех трёх вариантов полностью совпадают с учётом повторов.